In [1]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 11.3 MB/s eta 0:00:00


In [2]:
import warnings
from functools import partial

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV
from sklearn.experimental import enable_halving_search_cv
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.base import TransformerMixin, RegressorMixin, BaseEstimator
from sklearn.model_selection import TimeSeriesSplit, HalvingRandomSearchCV, train_test_split, GridSearchCV, ParameterGrid
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, make_scorer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping, lgb

import inspect
import optuna

import requests
from functools import reduce

from scipy.stats import loguniform, randint, uniform

from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

Mounted at /content/drive


In [3]:
def MirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_new = np.empty_like(X)
    X_new[X != 0] = np.sign(X[X != 0]) * (np.log(np.abs(X[X != 0]) * (1/c)) + np.log(c))
    X_new[X == 0] = 0
    return X_new

# reverses the mirror log transformation, restoring data—including negative or zero values—back to its original scale by undoing the signed logarithmic mapping.
def InverseMirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be inverse mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_inv = np.empty_like(X)
    X_inv[X != 0] = np.sign(X[X != 0]) * (np.exp(np.abs(X[X != 0]) - np.log(c)) - (1/c))
    X_inv[X == 0] = 0
    return X_inv


class MirrorLogNormScaler(TransformerMixin, BaseEstimator):

    def __init__(self, mirrorlog_kwargs:dict=None, normalizer=None):

        '''
        mirrorlog_kwargs    :   Keyword arguments for the mirror-log transformation. Default is {'c': 1/3}.
        normalizer          :   Normalizer to use after mirror-log transformation. Default is MinMaxScaler.
        '''

        self.mirrorlog_kwargs = mirrorlog_kwargs
        self.normalizer = normalizer
        _normalizer = self.normalizer or MinMaxScaler()
        self._normalizer = _normalizer

        mirrorlog_kwargs_ = self.mirrorlog_kwargs or {'c': 1/3}

        self._mlog_scaler = FunctionTransformer(
            func=partial(MirrorLog, **mirrorlog_kwargs_),
            inverse_func=partial(InverseMirrorLog, **mirrorlog_kwargs_),
            check_inverse=False
        )

    def fit(self, X, y=None):

        '''
            X   :   The data that will be mirror-log transformed then used to compute the per-feature minimum and maximum used for later scaling along the features axis.
            y   :   Ignored.
        '''

        X_mlog = self._mlog_scaler.fit_transform(X)
        self._normalizer.fit(X_mlog)
        return self

    def transform(self, X):

        '''
        X   :   The data that will be mirror-log transformed then min-max scaled.
        '''

        X_mlog = self._mlog_scaler.transform(X)
        X_new = self._normalizer.transform(X_mlog)
        return X_new

    def inverse_transform(self, X):

        '''
        X   :   The data that will be inverse min-max scaled then inverse mirror-log transformed.
        '''

        X_mlog = self._normalizer.inverse_transform(X)
        X_inv = self._mlog_scaler.inverse_transform(X_mlog)
        return X_inv


class ProcessingPipeline(RegressorMixin, BaseEstimator):

    '''
    rfecv_kwargs        :   Keyword arguments for the RFECV feature selection step. Estimator for feature importance must be provided. Default is a LGBM regressor with 3-fold CV and step size of 2.
    scaler              :   Scaler for feature normalization. Default is MinMaxScaler.
    estimator           :   Estimator for the regression task. Default is a LGBM regressor.
    target_transformer  :   Transformer (or scaler) for the target variable. If set to 'ignore', no transformation is applied. Default is MirrorLogNormScaler. Transformer must implement fit, transform, and inverse_transform methods.
    '''

    def __init__(self, rfecv_kwargs:dict=None, scaler=None, estimator=None, target_transformer=None):

        self.rfecv_kwargs = rfecv_kwargs
        self.scaler = scaler
        self.estimator = estimator
        self.target_transformer = target_transformer

        rfecv_kwargs = self.rfecv_kwargs or {'estimator':lgb.LGBMRegressor(verbose=-1), 'cv':3, 'step':2}
        scaler_ = self.scaler or MinMaxScaler()
        estimator_ = self.estimator or lgb.LGBMRegressor(verbose=-1)
        target_transformer_ = self.target_transformer or MirrorLogNormScaler()

        if self.rfecv_kwargs in (None, False, 'ignore', 'skip'):
            feature_step = ('feature_elimination', 'passthrough')
        else:
            feature_step = ('feature_elimination', RFECV(**self.rfecv_kwargs))

        pipe = Pipeline([
            ('feature_elimination', RFECV(**rfecv_kwargs)),
            ('normalization', scaler_),
            ('estimation', estimator_)
        ])

        if self.target_transformer != 'ignore':
            ttr_pipe = TransformedTargetRegressor(
                regressor=pipe,
                transformer=target_transformer_,
                check_inverse=False
            )
        else:
            ttr_pipe = None
        self.model = ttr_pipe if ttr_pipe else pipe

    def fit(self, X, y):

        '''
        X   :   Training matrix.
        y   :   Target values.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            self.model.fit(X, y)
        return self

    def predict(self, X):

        '''
        X   :   Samples.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            return self.model.predict(X)

def willmotts_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    wi = 1 - (np.sum((y_true-y_pred)**2) / np.sum((np.abs(y_pred-np.mean(y_true))+(np.abs(y_true-np.mean(y_pred))))**2))
    return wi

def nash_sutcliffe_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ns = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-np.mean(y_true))**2)
    return ns

def legates_mccabes_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    lm = 1 - np.sum(np.abs(y_pred-y_true)) / np.sum(np.abs(y_true-np.mean(y_true)))
    return lm

def kling_gupta_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cv_true = np.std(y_true) / np.mean(y_true)
    cv_pred = np.std(y_pred) / np.mean(y_pred)
    r = np.sum((y_true - y_true.mean()) * (y_pred - y_pred.mean())) / np.sqrt(np.sum((y_true - y_true.mean())**2) * np.sum((y_pred - y_pred.mean())**2))
    kge = 1 - np.sqrt((r-1)**2 + (np.mean(y_pred)/np.mean(y_true) - 1)**2 + (cv_pred/cv_true)**2)
    return kge

def normalized_root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    nrmse = root_mean_squared_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return nrmse

def relative_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmae = mean_absolute_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return rmae

def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    smape = (1/len(y_true)) * np.sum(np.abs(y_true-y_pred) / ((np.abs(y_true) + np.abs(y_pred))/2))
    return smape

def theils_inequality_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    numerator = np.sqrt((1/n) * (np.sum(y_pred-y_true)**2))
    denominator = np.sqrt((1/n) * np.sum(y_true**2)) + np.sqrt((1/n) * np.sum(y_pred**2))
    tic = numerator / denominator
    return tic

def absolute_percentage_bias(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    apb = np.abs(np.sum(y_true-y_pred) / np.sum(y_true))
    return apb

def evaluate_model(model, X, y_true) -> pd.DataFrame:
    '''
    model   :   A pre-trained model object.
    X       :   Feature matrix. Should be in the format your model object requires.
    y_true  :   True target array for prediction evaluation.
    '''

    if hasattr(model, 'predict'):
        y_pred = model.predict(X)
    elif callable(model):
        y_pred = model(X)
    else:
        raise TypeError('Model must be callable or have a .predict() method.')

    abbr = ['rmse', 'nrmse']
    prop = ['bias', 'bias']
    metrics = [root_mean_squared_error, normalized_root_mean_squared_error]

    results = [metric(y_true=y_true, y_pred=y_pred) for metric in metrics]

    df = pd.DataFrame(
        data=list(zip(prop, abbr, results)),
        columns=['property', 'metric', 'score']
    )

    return df

In [4]:
# Replace 'your_file_path.csv' with the actual path to your CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/MURES/rfe_dataset_2019_2025.csv'
df_rfe = pd.read_csv(csv_file_path)

# Setting Index
df_rfe['datetime'] = pd.to_datetime(df_rfe['datetime'], utc=True)
df_rfe = df_rfe.set_index('datetime')

y = df_rfe['price']
X_lag = df_rfe.drop(columns=['price'])

nrmse_scorer = make_scorer(normalized_root_mean_squared_error, greater_is_better=False)


In [5]:
# Removing NA values
df_xy = X_lag.copy()
df_xy["__y__"] = y
df_xy = df_xy.dropna()

y_clean = df_xy["__y__"]
X_clean = df_xy.drop(columns="__y__")

# Split the Data into Train and Test
X_train_val, X_test, y_train_val, y_test = train_test_split(X_clean, y_clean, test_size=0.1, shuffle=False)

train_size = int(len(X_train_val) * (0.9*0.8))  # 0.9 because train needs to be 80% of train/val dataset & validation shoudld be 90% of total dataset
step_size = 24*7*20  # every fold is 20 weeks in length
n_splits = (len(X_train_val) - train_size) // step_size
tscv = TimeSeriesSplit(n_splits=n_splits, max_train_size=train_size)

def _fit_supports(argname, estimator):
    """Check if the estimator.fit() method supports a given argument name."""
    sig = inspect.signature(estimator.fit)
    return argname in sig.parameters

In [6]:
## Hyperparameter tuning optimization using Optuna

def objective(trial):

    # XGBoost parameter
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_categorical("reg_alpha", [0.0, 1e-8, 1e-6, 1e-4, 1e-3]),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    # Cross validation on each trial + Calculate with early stopping method applied
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_val, y_train_val)):
        Xtr, Xva = X_train_val.iloc[tr_idx], X_train_val.iloc[va_idx]
        ytr, yva = y_train_val.iloc[tr_idx], y_train_val.iloc[va_idx]

        model = XGBRegressor(
            n_estimators=300,
            tree_method='hist',
            random_state=42,
            n_jobs= 1,
            objective='reg:squarederror',
            eval_metric='rmse',
            **params
        )

        fit_kwargs = dict(
            X=Xtr, y=ytr,
            eval_set=[(Xva, yva)],
            verbose=False
        )

        # EarlyStopping: Prioritize callbacks, if not use early_stopping_rounds
        if _fit_supports("callbacks", model):
            from xgboost.callback import EarlyStopping
            fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
        elif _fit_supports("early_stopping_rounds", model):
            fit_kwargs["early_stopping_rounds"] = 100

        model.fit(**fit_kwargs)

        # Using nrmse_scorer to calculate the performance on each fold
        score = nrmse_scorer(model, Xva, yva) if callable(getattr(nrmse_scorer, '__call__', None)) else float('nan')
        fold_scores.append(score)

        trial.report(float(np.nanmean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.nanmean(fold_scores))

# TPE + Hyperband Pruner
study = optuna.create_study(
    direction='maximize',  # better when score is higher
    sampler=optuna.samplers.TPESampler(seed=391),
    pruner=optuna.pruners.HyperbandPruner()
)

# Optimization
study.optimize(objective, n_trials=256, n_jobs=16)

print("[Optuna] Best score:", study.best_value)
print("[Optuna] Best params:", study.best_params)

best_params = study.best_params
best_score = study.best_value

# Final Model training using best parameters
final_model = XGBRegressor(
    n_estimators=1000,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse',
    **best_params
)

split = int(len(X_train_val) * 0.9)
fit_kwargs = dict(
    X=X_train_val.iloc[:split], y=y_train_val.iloc[:split],
    eval_set=[(X_train_val.iloc[split:], y_train_val.iloc[split:])],
    verbose=False
)
if _fit_supports("callbacks", final_model):
    from xgboost.callback import EarlyStopping
    fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
elif _fit_supports("early_stopping_rounds", final_model):
    fit_kwargs["early_stopping_rounds"] = 100

final_model.fit(**fit_kwargs)

class SearchLike:
    best_estimator_ = final_model
    best_params_    = best_params
    best_score_     = best_score
    study_          = study

search_es = SearchLike()
print("BEST:", search_es.best_score_, search_es.best_params_)



[I 2025-11-18 02:30:03,122] A new study created in memory with name: no-name-a81234fe-e25e-44ef-91c7-b4f5f85ef273
[I 2025-11-18 02:31:36,509] Trial 10 finished with value: -0.5504652835960462 and parameters: {'max_depth': 4, 'learning_rate': 0.046558210275273286, 'subsample': 0.8406346516186562, 'colsample_bytree': 0.6695069375568733, 'min_child_weight': 10, 'reg_alpha': 0.0001, 'reg_lambda': 0.4368268083658835}. Best is trial 10 with value: -0.5504652835960462.
[I 2025-11-18 02:32:12,577] Trial 2 finished with value: -0.5519276721080127 and parameters: {'max_depth': 5, 'learning_rate': 0.036129831809019165, 'subsample': 0.9112576295481863, 'colsample_bytree': 0.9650772772286571, 'min_child_weight': 5, 'reg_alpha': 0.0001, 'reg_lambda': 0.06007785717571588}. Best is trial 10 with value: -0.5504652835960462.
[I 2025-11-18 02:32:19,712] Trial 8 finished with value: -0.5792811354296296 and parameters: {'max_depth': 5, 'learning_rate': 0.004937810479975996, 'subsample': 0.9598491834664032,

[Optuna] Best score: -0.536529471308259
[Optuna] Best params: {'max_depth': 3, 'learning_rate': 0.04243714864677509, 'subsample': 0.6233752828875716, 'colsample_bytree': 0.9909653317759352, 'min_child_weight': 10, 'reg_alpha': 0.0, 'reg_lambda': 5.3877974642145645}
BEST: -0.536529471308259 {'max_depth': 3, 'learning_rate': 0.04243714864677509, 'subsample': 0.6233752828875716, 'colsample_bytree': 0.9909653317759352, 'min_child_weight': 10, 'reg_alpha': 0.0, 'reg_lambda': 5.3877974642145645}


In [7]:
evaluate_model(final_model, X_test, y_test)

,property,metric,score
0,bias,rmse,44.787713
1,bias,nrmse,0.513999


In [12]:
from lightgbm import LGBMRegressor, early_stopping

## Hyperparameter tuning optimization using Optuna (LightGBM)

def objective(trial):

    # LightGBM parameters
    params = {
      "num_leaves": trial.suggest_int("num_leaves", 31, 255),
      "max_depth": trial.suggest_int("max_depth", -1, 16),
      "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
      "subsample": trial.suggest_float("subsample", 0.6, 1.0),
      "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
      "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
      "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
      "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    # Cross validation on each trial + early stopping
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_val, y_train_val)):
        Xtr, Xva = X_train_val.iloc[tr_idx], X_train_val.iloc[va_idx]
        ytr, yva = y_train_val.iloc[tr_idx], y_train_val.iloc[va_idx]

        model = LGBMRegressor(
            n_estimators=1000,
            objective="regression",
            random_state=42,
            n_jobs=1,
            **params
        )

        fit_kwargs = dict(
            X=Xtr, y=ytr,
            eval_set=[(Xva, yva)],
            eval_metric="rmse",
        )

        # EarlyStopping
        if _fit_supports("callbacks", model):
            fit_kwargs["callbacks"] = [early_stopping(stopping_rounds=100)]
        elif _fit_supports("early_stopping_rounds", model):
            fit_kwargs["early_stopping_rounds"] = 100

        model.fit(**fit_kwargs)

        # Using nrmse_scorer to calculate the performance on each fold
        score = nrmse_scorer(model, Xva, yva) if callable(getattr(nrmse_scorer, '__call__', None)) else float('nan')
        fold_scores.append(score)

        trial.report(float(np.nanmean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.nanmean(fold_scores))


# TPE + Hyperband Pruner
study = optuna.create_study(
    direction='maximize',  # # better when score is higher
    sampler=optuna.samplers.TPESampler(seed=391),
    pruner=optuna.pruners.HyperbandPruner()
)

# Optimization
study.optimize(objective, n_trials=256, n_jobs=16)

print("[Optuna-LGBM] Best score:", study.best_value)
print("[Optuna-LGBM] Best params:", study.best_params)

best_params = study.best_params
best_score = study.best_value

# Final Model training using best parameters
final_model = LGBMRegressor(
    n_estimators=1500,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    **best_params
)

split = int(len(X_train_val) * 0.9)
fit_kwargs = dict(
    X=X_train_val.iloc[:split], y=y_train_val.iloc[:split],
    eval_set=[(X_train_val.iloc[split:], y_train_val.iloc[split:])],
    eval_metric="rmse"
)
if _fit_supports("callbacks", final_model):
    fit_kwargs["callbacks"] = [early_stopping(stopping_rounds=100)]
elif _fit_supports("early_stopping_rounds", final_model):
    fit_kwargs["early_stopping_rounds"] = 100

final_model.fit(**fit_kwargs)

class SearchLike:
    best_estimator_ = final_model
    best_params_    = best_params
    best_score_     = best_score
    study_          = study

search_es = SearchLike()
print("BEST (LGBM):", search_es.best_score_, search_es.best_params_)

[I 2025-11-18 03:43:58,612] A new study created in memory with name: no-name-c2656475-edfa-4fb8-9427-fdaac457d862


Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Early st

[I 2025-11-18 03:44:26,558] Trial 5 finished with value: -0.5361159179311066 and parameters: {'num_leaves': 144, 'max_depth': 3, 'learning_rate': 0.19701548442573374, 'subsample': 0.9908716220668878, 'colsample_bytree': 0.8361441820623228, 'min_child_samples': 66, 'reg_alpha': 0.00031624192999828583, 'reg_lambda': 0.16581521932819143}. Best is trial 5 with value: -0.5361159179311066.


Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[64]	valid_0's rmse: 26.227	valid_0's l2: 687.853
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[989]	valid_0's rmse: 157.314	valid_0's l2: 24747.8
Early stopping, best iteration is:
[97]	valid_0's rmse: 26.3975	valid_0's l2: 696.826
Early stopping, best iteration is:
[347]	valid_0's rmse: 26.3355	valid_0's l2: 693.559
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds


[I 2025-11-18 03:44:45,913] Trial 9 finished with value: -0.5362633875114955 and parameters: {'num_leaves': 190, 'max_depth': 1, 'learning_rate': 0.24416189051375015, 'subsample': 0.6555764665921484, 'colsample_bytree': 0.7570217342039128, 'min_child_samples': 92, 'reg_alpha': 0.0002281338808750599, 'reg_lambda': 0.038210243113972295}. Best is trial 5 with value: -0.5361159179311066.


...

Did not meet early stopping. Best iteration is:
[999]	valid_0's rmse: 153.869	valid_0's l2: 23675.8
Early stopping, best iteration is:
[80]	valid_0's rmse: 33.8196	valid_0's l2: 1143.76


[I 2025-11-18 04:06:47,525] Trial 253 pruned. 


Early stopping, best iteration is:
[55]	valid_0's rmse: 157.29	valid_0's l2: 24740.2
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds


[I 2025-11-18 04:06:48,336] Trial 254 finished with value: -0.5272404258507002 and parameters: {'num_leaves': 186, 'max_depth': 1, 'learning_rate': 0.14555168466025947, 'subsample': 0.6650010288523625, 'colsample_bytree': 0.8731885617284568, 'min_child_samples': 41, 'reg_alpha': 5.7761061094600283e-05, 'reg_lambda': 0.012832203732687751}. Best is trial 176 with value: -0.5248754661043161.
[I 2025-11-18 04:06:48,474] Trial 255 finished with value: -0.5263505202877905 and parameters: {'num_leaves': 43, 'max_depth': 1, 'learning_rate': 0.14019452232644736, 'subsample': 0.6122955883357384, 'colsample_bytree': 0.8724620419909139, 'min_child_samples': 41, 'reg_alpha': 5.362265104763842e-05, 'reg_lambda': 3.8799414732868454}. Best is trial 176 with value: -0.5248754661043161.


Early stopping, best iteration is:
[79]	valid_0's rmse: 33.6866	valid_0's l2: 1134.79
Early stopping, best iteration is:
[84]	valid_0's rmse: 33.4409	valid_0's l2: 1118.3
[Optuna-LGBM] Best score: -0.5248754661043161
[Optuna-LGBM] Best params: {'num_leaves': 182, 'max_depth': 1, 'learning_rate': 0.1376487510667218, 'subsample': 0.6646570414741191, 'colsample_bytree': 0.8579018466891751, 'min_child_samples': 48, 'reg_alpha': 2.5033847143506212e-05, 'reg_lambda': 3.188772872548122}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7320
[LightGBM] [Info] Number of data points in the train set: 42573, number of used features: 29
[LightGBM] [Info] Start training from score 99.681522
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further spl

In [13]:
evaluate_model(final_model, X_test, y_test)

,property,metric,score
0,bias,rmse,46.551135
1,bias,nrmse,0.534237
